# Regional (ROI) Analysis

Uses `roi_data.parquet` (produced by `regional_dataframe.ipynb`) to:

1. Identify the brain regions (ROIs) with the highest plaque burden
2. Compare plaque density and morphological metrics across treatment groups and genotypes
3. Summarise vessel-proximity fractions per ROI
4. Generate atlas heatmaps — max-intensity projections of metric volumes derived from
   `tpl-ABAv3_seg-all_dseg.nii.gz`


In [1]:
import warnings

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", font_scale=1.1)

TREAT_ORDER   = ["PBS", "Lecanemab"]
TREAT_PALETTE = {"PBS": "#4C72B0", "Lecanemab": "#DD8452"}
GENO_ORDER    = ["ApoE3", "ApoE4"]
GENO_PALETTE  = {"ApoE3": "#55A868", "ApoE4": "#C44E52"}

TOP_N = 20   # number of top regions to display in summary plots


## 1 · Load ROI data and atlas


In [2]:
roi_df = pd.read_parquet("roi_data.parquet")
roi_df["treatment"] = pd.Categorical(roi_df["treatment"], categories=TREAT_ORDER, ordered=True)
roi_df["genotype"]  = pd.Categorical(roi_df["genotype"],  categories=GENO_ORDER,  ordered=True)

# Remove regions without a valid volume (hierarchy-only nodes have volume_mm3 == 0)
roi_df = roi_df.loc[roi_df["volume_mm3"] > 0].copy()

print(f"Loaded {len(roi_df):,} subject-ROI rows  "
      f"({roi_df['subject'].nunique()} subjects, {roi_df['index'].nunique()} regions)")
roi_df.head(3)


Loaded 13,724 subject-ROI rows  (16 subjects, 1280 regions)


,subject,treatment,genotype,sex,index,name,plaque_count,total_vol_ml,mean_vol_ml,median_vol_ml,...,median_diam_um,total_vol_um3,mean_sdt_um,median_sdt_um,frac_inside_vessel,frac_near_vessel,frac_far_vessel,volume_mm3,plaque_density,vol_density_ml
0,sub-AS161F3,PBS,ApoE3,F,0,left root,30885,0.234113,0.000008,0.000003,...,18.355968,2.341129e+08,0.050244,-2.887072,0.646301,0.160855,0.192844,701.337955,44.037257,0.000334
1,sub-AS161F3,PBS,ApoE3,F,7,"left Frontal pole, layer 1",154,0.000696,0.000005,0.000002,...,16.765627,6.955238e+05,1.772841,-2.563844,0.577922,0.136364,0.285714,0.126750,1214.990080,0.005487
2,sub-AS161F3,PBS,ApoE3,F,8,"left Frontal pole, layer 2/3",18,0.000048,0.000003,0.000002,...,15.697833,4.809024e+04,-4.095661,-4.546432,0.944444,0.055556,0.000000,0.117031,153.805066,0.000411


In [3]:
atlas_img  = nib.load("tpl-ABAv3_seg-all_dseg.nii.gz")
atlas_data = atlas_img.get_fdata().astype(np.int32)
vox_mm     = atlas_img.header.get_zooms()[:3]          # voxel dimensions in mm
print(f"Atlas shape: {atlas_data.shape}  |  voxel size: {vox_mm} mm")


Atlas shape: (456, 528, 320)  |  voxel size: (np.float32(0.025), np.float32(0.025), np.float32(0.025)) mm


## 2 · Subject-mean ROI summary

Average each metric across subjects within each treatment × genotype group.


In [4]:
METRIC_COLS = [
    "plaque_count", "plaque_density", "vol_density_ml",
    "mean_diam_um", "median_diam_um",
    "mean_sdt_um",  "median_sdt_um",
    "frac_inside_vessel", "frac_near_vessel", "frac_far_vessel",
]

# Mean across subjects per region × group
roi_group = (
    roi_df
    .groupby(["index", "name", "treatment", "genotype"], observed=True)[METRIC_COLS]
    .mean()
    .reset_index()
)

# Overall mean across all subjects (ignoring group)
roi_mean = (
    roi_df
    .groupby(["index", "name"], observed=True)[METRIC_COLS]
    .mean()
    .reset_index()
)

print(f"Group-mean table: {len(roi_group):,} rows")
roi_group.head()


Group-mean table: 4,360 rows


,index,name,treatment,genotype,plaque_count,plaque_density,vol_density_ml,mean_diam_um,median_diam_um,mean_sdt_um,median_sdt_um,frac_inside_vessel,frac_near_vessel,frac_far_vessel
0,0,left root,PBS,ApoE3,29844.666667,42.553902,0.000321,21.068636,18.257576,0.136584,-2.682185,0.639850,0.167066,0.193083
1,0,left root,PBS,ApoE4,16289.000000,23.225607,0.000199,21.857419,18.927923,1.636478,-1.085394,0.546905,0.179886,0.273209
2,0,left root,Lecanemab,ApoE3,13903.000000,19.823539,0.000155,21.159956,18.333440,-0.021822,-1.509439,0.593875,0.200199,0.205926
3,0,left root,Lecanemab,ApoE4,16173.600000,23.061065,0.000194,21.725396,18.673942,1.316155,-1.688304,0.578149,0.176167,0.245685
4,7,"left Frontal pole, layer 1",PBS,ApoE3,207.000000,1633.136017,0.006502,18.216900,16.888843,-1.499808,-3.747814,0.777401,0.098930,0.123669


## 3 · Top ROIs by plaque density

Regions ranked by **mean plaque density** (plaques per mm³) across all subjects.


In [5]:
top_regions = (
    roi_mean
    .nlargest(TOP_N, "plaque_density")[
        ["index", "name", "plaque_density", "plaque_count", "vol_density_ml",
         "mean_diam_um", "frac_inside_vessel", "frac_near_vessel", "frac_far_vessel"]
    ]
    .reset_index(drop=True)
)
top_regions


,index,name,plaque_density,plaque_count,vol_density_ml,mean_diam_um,frac_inside_vessel,frac_near_vessel,frac_far_vessel
0,11,"left Frontal pole, layer 6b",1939.393847,1.000000,0.020630,27.286395,1.000000,0.000000,0.000000
1,7,"left Frontal pole, layer 1",1153.977591,146.266667,0.005310,20.493759,0.716028,0.125783,0.158189
2,10374,right Ectorhinal area/Layer 1,1069.520747,137.400000,0.005830,20.330373,0.743457,0.108807,0.147736
3,638,left Bed nucleus of the anterior commissure,1066.666616,4.000000,0.018114,25.013676,1.000000,0.000000,0.000000
4,10007,"right Frontal pole, layer 1",859.438305,108.866667,0.004120,19.794156,0.769910,0.125072,0.105018
5,10638,right Bed nucleus of the anterior commissure,790.123419,3.000000,0.002358,17.610804,1.000000,0.000000,0.000000
6,11141,right inferior colliculus commissure,773.747804,7.000000,0.005072,22.660583,0.142857,0.000000,0.857143
7,52,"left Primary somatosensory area, barrel field,...",771.306471,323.466667,0.006333,21.475813,0.502132,0.152002,0.345865
8,10368,"right Perirhinal area, layer 1",755.380246,83.800000,0.004884,21.236953,0.606717,0.150357,0.242926
9,374,left Ectorhinal area/Layer 1,727.401596,96.187500,0.003931,19.646927,0.684296,0.142738,0.172967


### 3a · Bar chart — top ROIs by mean plaque density


In [6]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(
    top_regions["name"],
    top_regions["plaque_density"],
    color="steelblue",
    edgecolor="white",
    height=0.7,
)
ax.set_xlabel("Mean plaque density (plaques / mm³)")
ax.set_title(f"Top {TOP_N} regions by plaque density (all subjects)")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("fig_roi_top_density.png", dpi=150)
plt.show()


## 4 · Plaque density by treatment and genotype (top ROIs)

Box plots showing the per-subject plaque density distribution for each of the top regions,
split by treatment and genotype.


In [7]:
top_idx  = top_regions["index"].tolist()
roi_top  = roi_df.loc[roi_df["index"].isin(top_idx)].copy()

# Short label: strip 'left '/'right ' prefix for display
roi_top["region_abbr"] = roi_top["name"].str.replace(r"^(left|right) ", "", regex=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=False)
for ax, geno in zip(axes, GENO_ORDER):
    sub = roi_top.loc[roi_top["genotype"] == geno]
    sns.boxplot(
        data=sub,
        y="region_abbr",
        x="plaque_density",
        hue="treatment",
        hue_order=TREAT_ORDER,
        palette=TREAT_PALETTE,
        orient="h",
        width=0.6,
        ax=ax,
    )
    ax.set_title(geno, fontsize=13)
    ax.set_xlabel("Plaque density (plaques / mm³)")
    ax.set_ylabel("Region")
    ax.legend(title="Treatment", loc="lower right")

fig.suptitle(f"Top {TOP_N} regions — plaque density by treatment", fontsize=14)
plt.tight_layout()
plt.savefig("fig_roi_density_boxplot.png", dpi=150)
plt.show()


## 5 · Multi-metric summary for top ROIs

Heatmap table showing the group-mean of all key metrics for each top region.
Each column is normalised to \[0, 1\] for visual comparison.


In [8]:
DISPLAY_METRICS = {
    "plaque_density":    "Density\n(n/mm³)",
    "vol_density_ml":    "Vol density\n(mL/mm³)",
    "mean_diam_um":      "Mean diam\n(µm)",
    "median_diam_um":    "Median diam\n(µm)",
    "mean_sdt_um":       "Mean SDT\n(µm)",
    "frac_inside_vessel":"Frac inside\nvessel",
    "frac_near_vessel":  "Frac near\nvessel",
    "frac_far_vessel":   "Frac far\nfrom vessel",
}

summary_tbl = (
    roi_mean
    .loc[roi_mean["index"].isin(top_idx)]
    .set_index("name")
    [list(DISPLAY_METRICS.keys())]
    .rename(columns=DISPLAY_METRICS)
    .loc[top_regions["name"]]   # preserve density-rank order
)
norm_tbl = (summary_tbl - summary_tbl.min()) / (
    summary_tbl.max() - summary_tbl.min() + 1e-12
)

fig, ax = plt.subplots(figsize=(13, 8))
im = ax.imshow(norm_tbl.values, cmap="YlOrRd", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(len(norm_tbl.columns)))
ax.set_xticklabels(norm_tbl.columns, fontsize=9)
ax.set_yticks(range(len(norm_tbl)))
ax.set_yticklabels(norm_tbl.index, fontsize=9)
plt.colorbar(im, ax=ax, label="Normalised value")
ax.set_title(
    f"Multi-metric summary — top {TOP_N} regions"
    " (all-subjects mean, normalised per column)"
)
plt.tight_layout()
plt.savefig("fig_roi_metric_heatmap.png", dpi=150)
plt.show()


## 6 · Vessel-proximity fractions by treatment (top ROIs)


In [9]:
prox_cols   = ["frac_inside_vessel", "frac_near_vessel", "frac_far_vessel"]
prox_labels = ["Inside vessel", "Near vessel", "Far from vessel"]

fig, axes = plt.subplots(1, len(prox_cols), figsize=(18, 7), sharey=True)
for ax, col, label in zip(axes, prox_cols, prox_labels):
    sns.boxplot(
        data=roi_top,
        y="region_abbr",
        x=col,
        hue="treatment",
        hue_order=TREAT_ORDER,
        palette=TREAT_PALETTE,
        orient="h",
        width=0.6,
        ax=ax,
    )
    ax.set_title(label, fontsize=12)
    ax.set_xlabel("Fraction of plaques")
    ax.set_ylabel("Region" if ax is axes[0] else "")
    ax.legend(title="Treatment", fontsize=8)

fig.suptitle(f"Vessel-proximity fractions — top {TOP_N} regions", fontsize=14)
plt.tight_layout()
plt.savefig("fig_roi_proximity_fractions.png", dpi=150)
plt.show()


## 7 · Treatment fold-change per ROI

Median fold-change (Lecanemab / PBS) for the top regions, split by genotype.
Colour scale: red = increased by Lecanemab, blue = reduced.


In [10]:
def roi_fold_change(roi_df, metric="plaque_density", top_idx=top_idx):
    """Return a (region × genotype) DataFrame of median fold-changes Lec / PBS."""
    sub  = roi_df.loc[roi_df["index"].isin(top_idx)].copy()
    rows = []
    for geno in GENO_ORDER:
        for _, row in top_regions.iterrows():
            pbs = sub.loc[
                (sub["genotype"] == geno) & (sub["treatment"] == "PBS") &
                (sub["index"] == row["index"]), metric
            ]
            lec = sub.loc[
                (sub["genotype"] == geno) & (sub["treatment"] == "Lecanemab") &
                (sub["index"] == row["index"]), metric
            ]
            fc = lec.median() / pbs.median() if pbs.median() > 0 else np.nan
            rows.append({"genotype": geno, "name": row["name"], "fold_change": fc})
    return pd.DataFrame(rows).pivot(index="name", columns="genotype", values="fold_change")


fc_df  = roi_fold_change(roi_df)
log_fc = np.log2(fc_df.values.astype(float))
vabs   = np.nanmax(np.abs(log_fc))

fig, ax = plt.subplots(figsize=(6, 8))
im = ax.imshow(log_fc, cmap="RdBu", aspect="auto", vmin=-vabs, vmax=vabs)
ax.set_xticks(range(len(fc_df.columns)))
ax.set_xticklabels(fc_df.columns, fontsize=11)
ax.set_yticks(range(len(fc_df)))
ax.set_yticklabels(fc_df.index, fontsize=9)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("log₂ fold-change (Lecanemab / PBS)")
ax.set_title("Treatment fold-change per ROI", fontsize=13)
plt.tight_layout()
plt.savefig("fig_roi_fold_change.png", dpi=150)
plt.show()


## 8 · Atlas heatmaps

For a given metric, each atlas voxel is coloured by its region’s mean value,
creating a volumetric heatmap.  
Three orthogonal **max-intensity projections** are displayed for each group.


In [11]:
def make_metric_volume(atlas_data, region_series):
    """
    Paint a metric onto the atlas volume using vectorised index lookup.

    Parameters
    ----------
    atlas_data    : int ndarray, shape (X, Y, Z) — atlas label volume
    region_series : pandas Series indexed by atlas region index (int),
                    values are the metric to paint

    Returns
    -------
    float32 ndarray of the same shape; unlisted regions are NaN.
    """
    max_idx = int(atlas_data.max()) + 1
    lookup  = np.full(max_idx, np.nan, dtype=np.float32)
    for ridx, val in region_series.items():
        if 0 <= int(ridx) < max_idx:
            lookup[int(ridx)] = float(val)
    vol = lookup[atlas_data]          # vectorised voxel-wise lookup
    vol[atlas_data == 0] = np.nan     # background → NaN
    return vol


def plot_atlas_heatmap(vol, title="", cmap="hot", cbar_label="",
                       vmin=None, vmax=None, savepath=None):
    """
    Display max-intensity projections along the three atlas axes.
    NaN voxels are shown as transparent (white background).
    """
    projs      = [np.nanmax(vol, axis=a) for a in range(3)]
    view_names = ["Projection along x", "Projection along y", "Projection along z"]

    _vmin = vmin if vmin is not None else np.nanmin(vol)
    _vmax = vmax if vmax is not None else np.nanmax(vol)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, proj, pname in zip(axes, projs, view_names):
        masked = np.ma.masked_invalid(proj)
        im = ax.imshow(
            masked.T, origin="lower", cmap=cmap,
            vmin=_vmin, vmax=_vmax, interpolation="nearest",
        )
        ax.set_title(pname, fontsize=10)
        ax.axis("off")

    plt.colorbar(im, ax=axes[-1], label=cbar_label, shrink=0.8)
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    if savepath:
        plt.savefig(savepath, dpi=150)
    plt.show()


### 8a · Overall plaque density heatmap (all subjects)


In [12]:
density_all = roi_df.groupby("index", observed=True)["plaque_density"].mean()

vol_density = make_metric_volume(atlas_data, density_all)
plot_atlas_heatmap(
    vol_density,
    title="Mean plaque density (all subjects)",
    cmap="hot",
    cbar_label="Plaque density (n / mm³)",
    vmin=0,
    savepath="fig_atlas_density_all.png",
)


### 8b · Plaque density heatmap by treatment × genotype


In [13]:
# Build volumes for every treatment × genotype combination
group_vols = {}
for geno in GENO_ORDER:
    for treat in TREAT_ORDER:
        grp_density = (
            roi_df
            .loc[(roi_df["genotype"] == geno) & (roi_df["treatment"] == treat)]
            .groupby("index", observed=True)["plaque_density"]
            .mean()
        )
        group_vols[(geno, treat)] = make_metric_volume(atlas_data, grp_density)

vmax_global = max(float(np.nanmax(v)) for v in group_vols.values())

n_rows, n_cols = len(GENO_ORDER), len(TREAT_ORDER) * 3
fig, axes_grid = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
if n_rows == 1:
    axes_grid = axes_grid[np.newaxis, :]

for gi, geno in enumerate(GENO_ORDER):
    for ti, treat in enumerate(TREAT_ORDER):
        v     = group_vols[(geno, treat)]
        projs = [np.nanmax(v, axis=a) for a in range(3)]
        for pi, proj in enumerate(projs):
            ax     = axes_grid[gi, ti * 3 + pi]
            masked = np.ma.masked_invalid(proj)
            im     = ax.imshow(
                masked.T, origin="lower", cmap="hot",
                vmin=0, vmax=vmax_global, interpolation="nearest",
            )
            ax.axis("off")
            if pi == 1:
                ax.set_title(f"{geno} / {treat}", fontsize=10)
            if ti == len(TREAT_ORDER) - 1 and pi == 2:
                plt.colorbar(im, ax=ax, label="n / mm³", shrink=0.8)

fig.suptitle("Plaque density heatmaps by treatment × genotype", fontsize=14)
plt.tight_layout()
plt.savefig("fig_atlas_density_groups.png", dpi=150)
plt.show()


### 8c · Treatment fold-change atlas heatmap

Per-region log₂ fold-change (Lecanemab / PBS) painted onto the atlas volume,
separately for each genotype.  
**Blue** = reduced by Lecanemab; **Red** = increased.  
Mean projection is used instead of max to preserve sign information.


In [14]:
fc_vols = {}
for geno in GENO_ORDER:
    pbs_mean = (
        roi_df
        .loc[(roi_df["genotype"] == geno) & (roi_df["treatment"] == "PBS")]
        .groupby("index", observed=True)["plaque_density"]
        .mean()
    )
    lec_mean = (
        roi_df
        .loc[(roi_df["genotype"] == geno) & (roi_df["treatment"] == "Lecanemab")]
        .groupby("index", observed=True)["plaque_density"]
        .mean()
    )
    common  = pbs_mean.index.intersection(lec_mean.index)
    log2fc  = np.log2(
        (lec_mean.loc[common] + 1e-9) / (pbs_mean.loc[common] + 1e-9)
    )
    fc_vols[geno] = make_metric_volume(atlas_data, log2fc)

vabs_global = max(
    float(np.nanmax(np.abs(v[~np.isnan(v)]))) for v in fc_vols.values()
)

fig, axes_grid = plt.subplots(len(GENO_ORDER), 3,
                               figsize=(14, 4 * len(GENO_ORDER)))
if len(GENO_ORDER) == 1:
    axes_grid = axes_grid[np.newaxis, :]

for gi, geno in enumerate(GENO_ORDER):
    v     = fc_vols[geno]
    projs = [np.nanmean(v, axis=a) for a in range(3)]
    for pi, proj in enumerate(projs):
        ax     = axes_grid[gi, pi]
        masked = np.ma.masked_invalid(proj)
        im     = ax.imshow(
            masked.T, origin="lower", cmap="RdBu",
            vmin=-vabs_global, vmax=vabs_global, interpolation="nearest",
        )
        ax.axis("off")
        if pi == 1:
            ax.set_title(geno, fontsize=11)
    plt.colorbar(
        im, ax=axes_grid[gi, -1],
        label="log₂ FC (Lec / PBS)", shrink=0.8,
    )

fig.suptitle("Treatment fold-change (Lecanemab / PBS) — plaque density", fontsize=14)
plt.tight_layout()
plt.savefig("fig_atlas_fold_change.png", dpi=150)
plt.show()


### 8d · Mean plaque diameter heatmap (all subjects)


In [15]:
diam_all = roi_df.groupby("index", observed=True)["mean_diam_um"].mean()

vol_diam = make_metric_volume(atlas_data, diam_all)
plot_atlas_heatmap(
    vol_diam,
    title="Mean plaque diameter (all subjects)",
    cmap="plasma",
    cbar_label="Mean equivalent diameter (µm)",
    vmin=0,
    savepath="fig_atlas_mean_diam.png",
)


### 8e · Fraction of plaques inside vessels heatmap (all subjects)


In [16]:
inside_all = roi_df.groupby("index", observed=True)["frac_inside_vessel"].mean()

vol_inside = make_metric_volume(atlas_data, inside_all)
plot_atlas_heatmap(
    vol_inside,
    title="Fraction of plaques inside vessels (all subjects)",
    cmap="viridis",
    cbar_label="Fraction inside vessel",
    vmin=0,
    savepath="fig_atlas_frac_inside.png",
)
